## Model Training


In [114]:
import pandas as pd
df=pd.read_csv("data/gemstone.csv")
df

,id,carat,cut,color,clarity,depth,table,x,y,z,price
0,0,1.52,Premium,F,VS2,62.2,58.0,7.27,7.33,4.55,13619
1,1,2.03,Very Good,J,SI2,62.0,58.0,8.06,8.12,5.05,13387
2,2,0.70,Ideal,G,VS1,61.2,57.0,5.69,5.73,3.50,2772
3,3,0.32,Ideal,G,VS1,61.6,56.0,4.38,4.41,2.71,666
4,4,1.70,Premium,G,VS2,62.6,59.0,7.65,7.61,4.77,14453
...,...,...,...,...,...,...,...,...,...,...,...
193568,193568,0.31,Ideal,D,VVS2,61.1,56.0,4.35,4.39,2.67,1130
193569,193569,0.70,Premium,G,VVS2,60.3,58.0,5.75,5.77,3.47,2874
193570,193570,0.73,Very Good,F,SI1,63.1,57.0,5.72,5.75,3.62,3036
193571,193571,0.34,Very Good,D,SI1,62.9,55.0,4.45,4.49,2.81,681


In [115]:
### Drop ID column as it is not that important

df.drop(labels=['id'],axis=1,inplace=True)
df

,carat,cut,color,clarity,depth,table,x,y,z,price
0,1.52,Premium,F,VS2,62.2,58.0,7.27,7.33,4.55,13619
1,2.03,Very Good,J,SI2,62.0,58.0,8.06,8.12,5.05,13387
2,0.70,Ideal,G,VS1,61.2,57.0,5.69,5.73,3.50,2772
3,0.32,Ideal,G,VS1,61.6,56.0,4.38,4.41,2.71,666
4,1.70,Premium,G,VS2,62.6,59.0,7.65,7.61,4.77,14453
...,...,...,...,...,...,...,...,...,...,...
193568,0.31,Ideal,D,VVS2,61.1,56.0,4.35,4.39,2.67,1130
193569,0.70,Premium,G,VVS2,60.3,58.0,5.75,5.77,3.47,2874
193570,0.73,Very Good,F,SI1,63.1,57.0,5.72,5.75,3.62,3036
193571,0.34,Very Good,D,SI1,62.9,55.0,4.45,4.49,2.81,681


In [116]:
### independent and dependent dataframe

X=df.drop(labels=['price'],axis=1)
Y=df[['price']]

In [117]:
X



,carat,cut,color,clarity,depth,table,x,y,z
0,1.52,Premium,F,VS2,62.2,58.0,7.27,7.33,4.55
1,2.03,Very Good,J,SI2,62.0,58.0,8.06,8.12,5.05
2,0.70,Ideal,G,VS1,61.2,57.0,5.69,5.73,3.50
3,0.32,Ideal,G,VS1,61.6,56.0,4.38,4.41,2.71
4,1.70,Premium,G,VS2,62.6,59.0,7.65,7.61,4.77
...,...,...,...,...,...,...,...,...,...
193568,0.31,Ideal,D,VVS2,61.1,56.0,4.35,4.39,2.67
193569,0.70,Premium,G,VVS2,60.3,58.0,5.75,5.77,3.47
193570,0.73,Very Good,F,SI1,63.1,57.0,5.72,5.75,3.62
193571,0.34,Very Good,D,SI1,62.9,55.0,4.45,4.49,2.81


In [118]:
Y

,price
0,13619
1,13387
2,2772
3,666
4,14453
...,...
193568,1130
193569,2874
193570,3036
193571,681


In [119]:
### categorical columns

X.dtypes[X.dtypes=='object'].index

Index(['cut', 'color', 'clarity'], dtype='object')

In [120]:
df[X.columns[X.dtypes=='object']]

,cut,color,clarity
0,Premium,F,VS2
1,Very Good,J,SI2
2,Ideal,G,VS1
3,Ideal,G,VS1
4,Premium,G,VS2
...,...,...,...
193568,Ideal,D,VVS2
193569,Premium,G,VVS2
193570,Very Good,F,SI1
193571,Very Good,D,SI1


In [121]:
### Define which columns should be ordinal-encoded and which should be scaled. 
categorical_cols=X.select_dtypes(include='object').columns
type(categorical_cols)

numerical_cols=X.select_dtypes(exclude='object').columns
numerical_cols



Index(['carat', 'depth', 'table', 'x', 'y', 'z'], dtype='object')

In [122]:
### Define the custom ranking for each individual variable. It is in the order
cut_categories = ['Fair','Good','Very Good','Premium','Ideal']
color_categories = ['D','E','F','G','H','I','J']
clarity_categories = ['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF']
X['cut'].unique()

array(['Premium', 'Very Good', 'Ideal', 'Good', 'Fair'], dtype=object)

In [123]:
### this will do same functionality to which df.fillna does. automating all EDA processes
from sklearn.impute import SimpleImputer ### Handling missing values
from sklearn.preprocessing import StandardScaler ### Handling Feature Scaling
from  sklearn.preprocessing import OrdinalEncoder  ### Handling Feature Engineering(Ordinal Encoding). wherever we have categorical ordinal we use ordinal encoding, otherwise we would have used onhot encoding
### pipelines .it is just to combining multiple steps

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer ### combile numerical and categorical pipeline



In [124]:
#### Numerical Pipeline
num_pipeline=Pipeline(
    steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
    ]
)

#### categorical pnum_pipeline
cat_pipeline=Pipeline(
    steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('ordinalencoder',OrdinalEncoder(categories=[cut_categories,color_categories,clarity_categories])),
    ('scaler',StandardScaler())

    ]
)

#combine both pipelines with column tColumnTransformer

preprocessor = ColumnTransformer([
    ('num_pipeline',num_pipeline,numerical_cols),
    ('cat_pipeline',cat_pipeline,categorical_cols)

]
)

In [125]:
#### Train Test Split

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,Y,test_size=0.30,random_state=42)


In [126]:
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        print(f"{col} unique values: {X_train[col].unique()}")

cut unique values: ['Ideal' 'Very Good' 'Premium' 'Good' 'Fair']
color unique values: ['E' 'H' 'F' 'I' 'G' 'D' 'J']
clarity unique values: ['VVS2' 'VS1' 'VS2' 'VVS1' 'SI1' 'SI2' 'IF' 'I1']


In [127]:
#preprocessor.fit_transform(X_train)
X_train=pd.DataFrame(preprocessor.fit_transform(X_train),columns=preprocessor.get_feature_names_out())

In [128]:
#preprocessor.transform(X_test)
X_test=pd.DataFrame(preprocessor.transform(X_test),columns=preprocessor.get_feature_names_out())

In [129]:
#### Model Training 
from sklearn.linear_model import LinearRegression,Lasso,Ridge,ElasticNet
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [130]:
#### Regression model 

regression =LinearRegression()
regression.fit(X_train,y_train)

LinearRegression()

In [131]:
regression.coef_

array([[ 6432.97591819,  -132.34206204,   -70.48787525, -1701.38593925,
         -494.17005097,   -76.32351645,    68.80035873,  -464.67990411,
          652.10059539]])

In [132]:
regression.intercept_

array([3976.8787389])

In [113]:
import numpy as np

def evaluate_model(true,predicted):
    mae=mean_absolute_error(true,predicted)
    mse=mean_squared_error(true,predicted)
    rmse=np.sqrt(mean_squared_error(true,predicted))
    r2_square=r2_score(true,predicted)
    return mae, rmse, r2_square
    

In [ ]:
#### Train Multiple models 

models= {
    'LinearRegression': LinearRegression(),
    'Lasso' : Lasso(),
    'Ridge' : Ridge(),
    'Elasticnet' : ElasticNet()
}

model_List=[]

r2_list=[]

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)

    #make prediction 
    y_pred=model.predict(X_test)
    mae, rmse, r2_square = evaluate_model(y_test,y_pred)

    print(list(models.keys())[i])
    model_List.append(list(models.keys())[i])

    print("Model Training Performance")
    print("RMSE:",rmse)
    print("MAE:",mae)
    print("R2 Score:",r2_square)

    r2_list.append(r2_square)

    print("="*35)
    print('\n')


    #### sort the r2_score, whichever has the highest value select that model



LinearRegression
Model Training Performance
RMSE: 1014.6296630375463
MAE: 675.0758270067483
R2 Score: 0.9362906819996049


Lasso
Model Training Performance
RMSE: 1014.659130275064
MAE: 676.2421173665508
R2 Score: 0.9362869814082755


Ridge
Model Training Performance
RMSE: 1014.6343233534411
MAE: 675.1077629781329
R2 Score: 0.9362900967491632


Elasticnet
Model Training Performance
RMSE: 1533.3541245902313
MAE: 1060.9432977143008
R2 Score: 0.8544967219374031


